In [ ]:
import pandas as pd
import os

# 读取原始CSV文件
file_path = r'E:\\NYPD_Arrests_Data_(Historic)_20251024.csv'
try:
    # 尝试用不同编码读取
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='latin1')  # 使用更广泛的编码
    
    print(f"成功读取数据，总记录数: {len(df)}")
    
    # 打印前5条记录的日期值（用于验证格式）
    print("\n前5条记录的ARREST_DATE值 (格式应为 MM/DD/YYYY):")
    print(df['ARREST_DATE'].head(5).to_string(index=False))
    
    # 使用pd.to_datetime直接解析日期（指定格式为 %m/%d/%Y）
    df['ARREST_DATE'] = pd.to_datetime(
        df['ARREST_DATE'],
        format='%m/%d/%Y',  # 对应 MM/DD/YYYY 格式
        errors='coerce'      # 无效日期转为NaT
    )
    
    # 检查无效日期记录
    invalid_dates = df['ARREST_DATE'].isna().sum()
    if invalid_dates > 0:
        print(f"\n警告: 发现 {invalid_dates} 条记录的日期无效（占比 {invalid_dates/len(df)*100:.2f}%）")
        
        # 显示部分无效日期示例
        print("\n无效日期示例:")
        invalid_samples = df[df['ARREST_DATE'].isna()]['ARREST_DATE'].index[:5]
        for idx in invalid_samples:
            original_value = df.loc[idx, 'ARRESTE_DAT']
            print(f"  行 {idx}: '{original_value}'")
    
    # 提取年份
    df['YEAR'] = df['ARREST_DATE'].dt.year
    
    # 删除年份缺失的记录
    initial_count = len(df)
    df = df.dropna(subset=['YEAR'])
    cleaned_count = len(df)
    
    if initial_count > cleaned_count:
        print(f"已移除 {initial_count - cleaned_count} 条无效日期记录")
    
    # 确保年份为整数
    df['YEAR'] = df['YEAR'].astype(int)
    
    # 检查数据年份范围
    min_year = df['YEAR'].min()
    max_year = df['YEAR'].max()
    print(f"\n有效年份范围: {min_year} - {max_year}")
    
    # 创建输出目录
    output_dir = r'E:\\'
    os.makedirs(output_dir, exist_ok=True)

    # 按年份分割保存文件
    print(f"\n开始导出数据...")
    for year in range(min_year, max_year + 1):
        year_df = df[df['YEAR'] == year]
        
        if not year_df.empty:
            output_path = os.path.join(output_dir, f'arrests_{year}.csv')
            # 移除添加的年份列
            year_df.drop(columns=['YEAR']).to_csv(
                output_path, 
                index=False,
                encoding='utf-8-sig'
            )
            print(f"已保存 {year} 年数据: {len(year_df)} 条记录 → {output_path}")
        else:
            print(f"警告: {year} 年无数据记录")
    
    print("\n" + "="*50)
    print(f"处理完成! 总有效记录数: {len(df)}")
    print(f"导出了 {max_year - min_year + 1} 个年份的数据")
    print(f"输出目录: {output_dir}")
    
except Exception as e:
    print(f"\n处理过程中出错: {str(e)}")
    import traceback
    traceback.print_exc()
